# ⚙️ Módulos y Librerías: Lectura de Bajo Nivel y Decodificación (`os.read` y `decode`)

| Propiedad | Detalle |
| :--- | :--- |
| 📍 **Módulo** | Módulos, Paquetes y Librerías / Módulo OS |
| 🎯 **Objetivo** | Entender el flujo de lectura directa mediante `os.read()` utilizando descriptores de bajo nivel, procesar búferes de bytes (`bytes`) y realizar la conversión a cadenas de texto (`str`) mediante `.decode()`. |
| 🛠️ **Conceptos** | `os.read()`, `bytes`, `decode()`, `Encoding (utf-8, latin-1)`, `File Descriptor`, `Búferes de memoria` |

---

### 💡 Lectura de Ficheros en Bajo Nivel con `os.read()`

La función **`os.read(fd, n)`** lee como máximo $n$ bytes desde el descriptor de archivo especificado (`fd`). A diferencia de las funciones de alto nivel (`file.read()`), esta llamada del sistema operativo no procesa texto directamente: retorna un objeto de tipo **`bytes`** nativo.

---

### 🗂️ Flujo de Conversión de Datos (Bytes a Texto)

Para transformar la secuencia de bytes crudos a una cadena legible en Python, se utiliza el método **`.decode()`**:

```text
[ Disco Duro / SO ] ──( os.read )──> [ Buffer de Bytes: b'...' ] ──( .decode('utf-8') )──> [ Cadenas de Texto: "..." ]
```

| Componente | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| 🔢 `os.read(fd, length)` | `bytes` | Lee los datos binarios directamente de la memoria sin interpretar la codificación. |
| 🔤 `.decode(encoding)` | `str` | Interpreta el flujo de bytes usando una tabla de caracteres (ej. UTF-8) para obtener texto. |
```

In [ ]:
import os
from typing import Any

# 🛰️ [1] Banner visual de inicio de ejecución
print("=" * 80)
print("🐍 MÓDULO OS | Lectura en Bajo Nivel con os.read() y Decodificación decode()")
print("=" * 80)

# ================================================================================
# 🧪 1. LECTURA BÁSICA DE BYTES Y DECODIFICACIÓN A TEXTO
# ================================================================================
# Lectura mediante descriptor entero y conversión usando utf-8.
# ================================================================================

print("\n--- 🚀 [PASO 1]: Uso de os.read() y decode() ---")

sample_file_path: str = "low_level_reading.txt"

# Creación previa del archivo con caracteres especiales para probar la codificación
with open(sample_file_path, "w", encoding="utf-8") as setup_file:
    setup_file.write("Python OS Module: Lectura de bajo nivel con caracteres UTF-8 (á, é, ñ).")

file_descriptor: int | None = None

try:
    # 🔌 [1] Apertura del archivo en bajo nivel (Modo Solo Lectura)
    file_descriptor = os.open(sample_file_path, os.O_RDONLY)
    print(f"📋 Descriptor de archivo asignado: {file_descriptor}")

    # 🔌 [2] Lectura de los primeros 100 bytes del archivo
    buffer_bytes: bytes = os.read(file_descriptor, 100)
    print(f"  🔹 Bytes crudos leídos ({len(buffer_bytes)} bytes): {buffer_bytes}")

    # 🔌 [3] Decodificación del buffer de bytes a cadena de texto
    decoded_text: str = buffer_bytes.decode("utf-8")
    print(f"  ✅ Texto decodificado exitosamente:\n     '{decoded_text}'")

finally:
    # 🔌 [4] Liberación obligatoria del recurso
    if file_descriptor is not None:
        os.close(file_descriptor)
        print("  🔒 Descriptor cerrado mediante os.close().")

    # Limpieza del archivo de demostración
    if os.path.exists(sample_file_path):
        os.remove(sample_file_path)

## 🧱 Lectura Fragmentada por Bloques (*Chunking*)

### 💡 Procesamiento de Archivos Grandes:
Cuando el archivo supera la capacidad de memoria disponible, se pueden realizar lecturas consecutivas invocando `os.read()` en un bucle `while`. Cada llamada avanzará automáticamente la posición del puntero dentro del descriptor hasta retornar un objeto `b''` (búfer vacío que indica el fin del archivo / EOF).

In [ ]:
# ================================================================================
# 🧪 2. LECTURA EN BÚFERES ITERATIVOS (CHUNKS)
# ================================================================================
print("\n--- 🚀 [PASO 2]: Lectura Secuencial por Chunks de Bytes ---")

chunk_file_path: str = "chunk_reading_demo.txt"

# Creación de un archivo más extenso
with open(chunk_file_path, "w", encoding="utf-8") as setup_file:
    setup_file.write("LINEA_1: Transacción Alpha completada.\n")
    setup_file.write("LINEA_2: Transacción Beta procesada.\n")
    setup_file.write("LINEA_3: Transacción Gamma finalizada.\n")

chunk_fd: int | None = None

try:
    chunk_fd = os.open(chunk_file_path, os.O_RDONLY)
    chunk_size: int = 32  # Tamaño de bloque reducido a 32 bytes para la demostración
    
    print(f"📋 Leyendo archivo en bloques de {chunk_size} bytes:")
    block_index: int = 1

    while True:
        # ⚙️ [1] Lectura del fragmento de bytes
        raw_chunk: bytes = os.read(chunk_fd, chunk_size)
        
        # Si el resultado es b'', se alcanzó el final del archivo (EOF)
        if not raw_chunk:
            print("  🏁 Fin de archivo alcanzado (EOF).")
            break

        # ⚙️ [2] Decodificación del bloque actual
        text_chunk: str = raw_chunk.decode("utf-8", errors="replace")
        print(f"  🔹 Bloque {block_index:02d} | Bytes: {len(raw_chunk):02d} | Contenido: {repr(text_chunk)}")
        block_index += 1

finally:
    if chunk_fd is not None:
        os.close(chunk_fd)
        print("  🔒 Descriptor liberado exitosamente.")

    if os.path.exists(chunk_file_path):
        os.remove(chunk_file_path)

## 🛒 Ejercicios Prácticos Avanzados Integrados

---

### 🧪 Ejercicio 1: Lector y Decodificador Seguro de Archivos Binarios / Logs

**Escenario Empresarial:** Se necesita implementar un componente de seguridad que abra un archivo en bajo nivel, extraiga los primeros $N$ bytes del encabezado (*header*) para auditar los metadatos y gestione errores de decodificación si se encuentran caracteres corruptos o no compatibles con UTF-8.

In [ ]:
# ================================================================================
# 🧪 3. EJERCICIO PRÁCTICO 1: INSPECTOR Y DECODIFICADOR SEGURO
# ================================================================================
print("\n--- 🧪 [PASO 3.1]: Ejercicio 1 - Extractor Seguro de Encabezados ---")


def read_and_decode_file_header(
    file_path: str,
    byte_limit: int = 64,
    encoding: str = "utf-8"
) -> dict[str, Any]:
    """Abre un archivo en bajo nivel, lee un límite de bytes y los decodifica de forma segura.

    Args:
        file_path (str): Ruta del archivo a inspeccionar.
        byte_limit (int): Número máximo de bytes a leer del inicio.
        encoding (str): Algoritmo de decodificación a emplear.

    Returns:
        dict[str, Any]: Reporte del resultado de lectura y decodificación.
    """
    execution_result: dict[str, Any] = {
        "success": False,
        "bytes_read": 0,
        "decoded_content": "",
        "error": None
    }

    file_descriptor: int | None = None

    try:
        # [1] Intentar la apertura en modo solo lectura
        file_descriptor = os.open(file_path, os.O_RDONLY)
        
        # [2] Lectura del flujo de bytes de bajo nivel
        raw_bytes: bytes = os.read(file_descriptor, byte_limit)
        execution_result["bytes_read"] = len(raw_bytes)

        # [3] Decodificación segura tolerando errores de formato
        execution_result["decoded_content"] = raw_bytes.decode(encoding, errors="replace")
        execution_result["success"] = True
        print(f"  ✅ Lectura completada: {len(raw_bytes)} bytes obtenidos.")

    except OSError as os_err:
        execution_result["error"] = f"Error del SO al leer el fichero: {os_err}"
        print(f"  ❌ {execution_result['error']}")

    finally:
        # [4] Liberación del descriptor dentro de la cláusula finally
        if file_descriptor is not None:
            os.close(file_descriptor)
            print("  🔒 Descriptor de archivo liberado correctamente.")

    return execution_result


# Prueba del ejercicio
audit_log_path: str = "binary_header_audit.log"

with open(audit_log_path, "w", encoding="utf-8") as test_file:
    test_file.write("HEADER_V1_SYSTEM_STATION_ALPHA_2026_STATUS_OK\nDATOS...")

print(f"📋 Ejecutando extracción de encabezado en bajo nivel sobre '{audit_log_path}'...")
audit_output: dict[str, Any] = read_and_decode_file_header(audit_log_path, byte_limit=45)

print("\n📋 Resumen del Informe de Inspección:")
print(f"  🔹 Operación exitosa:   {audit_output['success']}")
print(f"  🔹 Total bytes leídos:  {audit_output['bytes_read']}")
print(f"  🔹 Texto decodificado:  '{audit_output['decoded_content']}'")

# Limpieza del archivo de prueba
if os.path.exists(audit_log_path):
    os.remove(audit_log_path)

# --------------------------------------------------------------------------------
# 🏁 [PASO 4]: Cierre de Ejecución
# --------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("✅ [FIN DE EJECUCIÓN] Lección de os.read() y decode() finalizada.")
print("=" * 80)